In [77]:
print('Hello World')

Hello World


In [79]:
!nvidia-smi

Sat Aug  8 12:39:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:4A:00.0 Off |                    0 |
| N/A   39C    P0            102W /  350W |     545MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [80]:
import os
import sys
import torch
from absl import app, flags
from tqdm import tqdm
import torchvision


In [81]:
original_images=torch.load('/home/saurabhg/scratch/cifar10_data/image_tensor/original_image_100.pt')
generated_images=torch.load('/home/saurabhg/scratch/flow_outputs/icfm/image_tensor/generated_ICFM_images_step_90000.pt')
indices=torch.load('/home/saurabhg/scratch/flow_outputs/icfm/distances/lpips_minimum_indices.pt')

In [82]:
generated_images
original_images=original_images.to(generated_images.device)

In [84]:
generated_images.device

device(type='cuda', index=0)

In [85]:
indices

tensor([[4.3000e+01, 1.1906e-03],
        [2.0000e+01, 7.7082e-04],
        [1.2400e+02, 2.1281e-03],
        [1.5800e+02, 2.0187e-03],
        [8.3000e+01, 3.9655e-01],
        [9.0000e+01, 4.6670e-01],
        [1.7200e+02, 3.9861e-01],
        [2.1000e+01, 5.0646e-01],
        [1.1800e+02, 7.3679e-04],
        [1.1500e+02, 4.5995e-01],
        [1.3000e+02, 1.7246e-03],
        [1.5000e+01, 1.0855e-03],
        [5.8000e+01, 8.3726e-04],
        [3.1000e+01, 4.5025e-01],
        [4.4000e+01, 7.9372e-04],
        [1.0200e+02, 4.2923e-01],
        [1.0200e+02, 1.2258e-03],
        [1.9300e+02, 9.1472e-04],
        [5.2000e+01, 3.8590e-01],
        [1.6500e+02, 1.1873e-03],
        [5.0000e+01, 6.4836e-04],
        [1.0400e+02, 1.0185e-03],
        [1.0000e+00, 3.9360e-01],
        [1.0000e+02, 3.9544e-01],
        [3.4000e+01, 1.1100e-03],
        [8.1000e+01, 7.7949e-04],
        [8.1000e+01, 4.5522e-01],
        [1.4000e+02, 2.9271e-03],
        [1.1100e+02, 6.4661e-04],
        [2.300

In [86]:
sorted_distance=torch.sort(indices[:,1])
sorted_distances_indices=sorted_distance[1]


In [87]:
closest_generated_images=generated_images[indices[:,0].long()]

In [96]:
compare_image=torch.stack((original_images,closest_generated_images),dim=-1)

In [97]:
compare_image=compare_image[sorted_distances_indices].permute([0,4,1,2,3]).reshape(-1,3,32,32)


In [98]:
compare_image.shape

torch.Size([200, 3, 32, 32])

In [100]:
torchvision.utils.save_image(compare_image,'/home/saurabhg/scratch/flow_outputs/icfm/distances/compare_images_sorted_distance.png',nrow=2)

In [103]:
!python /home/saurabhg/SFU_research_internship/examples/images/cifar10/metric_distance.py

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/saurabhg/flow_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/saurabhg/flow_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/saurabhg/flow_env/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth
100%|███████████████████████████████████████| 100/100 [00:00<00:00, 130.86it/s]
Done


In [108]:
torchvision.utils.save_image(generated_images,'/home/saurabhg/scratch/flow_outputs/icfm/generated_images/200_images_generated.png',nrow=10)